# 3장. 다음 글자 맞히기 문제와 임베딩

글에서 '다음 글자 맞히기' 문제를 만들고, 글자 번호를 모델이 계산할 수 있는 벡터로 바꿔요. 이 장은 **GPU가 필요 없어요.** (런타임 유형: CPU)

이 장에서 PyTorch의 **텐서**, **모양(shape)**, **`nn.Embedding`**, 그리고 이 책의 첫 **클래스**를 만납니다.


### 준비
2장에서 만든 것과 같은 사전을 `Vocab.build()`로 만들고, 이야기를 불러옵니다.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from torch import nn
from modu_llm import Vocab, load_stories, END

torch.manual_seed(42)
vocab = Vocab.build()
stories = load_stories(3000)
text = END.join(stories) + END
print("사전 크기:", len(vocab))
print("글자 수:", len(text))


## 3.1 정답이 필요 없는 문제: 한 칸 밀기

글 자체가 문제이자 정답입니다. 입력 x를 한 칸 뒤로 밀면 정답 y가 돼요.
x의 각 위치에서 "여기까지 보고 다음 글자를 맞혀라"라는 문제가 하나씩 생깁니다.

**핵심 코드**: `y = data[1:block_size + 1]` — 정답은 입력보다 한 칸 뒤


In [ ]:
data = torch.tensor(vocab.encode(text))
block_size = 8

x = data[:block_size]
y = data[1:block_size + 1]

for t in range(block_size):
    context = vocab.decode(x[:t + 1].tolist())
    target = vocab.decode([y[t].item()])
    print(f"[{context}] 다음 → [{target}]")


## 3.2 학습용과 검증용 나누기

90%는 학습에, 10%는 검증에 씁니다. 검증용 글은 학습에 쓰지 않고, **처음 보는 글에서도 잘 맞히는지** 확인하는 데만 써요.


In [ ]:
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

print("학습용:", len(train_data))
print("검증용:", len(val_data))


## 3.3 텐서와 모양 읽기

**텐서**는 숫자를 담는 상자예요. 줄로 늘어놓으면 1차원, 표로 놓으면 2차원, 표를 여러 장 겹치면 3차원입니다.
**모양(shape)**은 각 방향으로 숫자가 몇 개인지 알려 줍니다.

**핵심 코드**: `data.shape`


In [ ]:
print("data:", data.shape, data.dtype)

table = data[:12].view(3, 4)
print("표로 바꾸면:", table.shape)
print(table)


## 3.4 여러 문제를 한 번에: 미리 보기

3.1에서는 글 맨 앞에서 조각 하나만 잘랐어요. 실제 학습에서는 글의 여기저기에서 조각을 여러 개 잘라 **쌓아서** 한꺼번에 풀게 합니다.
이 묶음을 **배치**라고 해요. 배치는 7장에서 GPT를 조립할 때 다시 만납니다. 4~6장은 문장 하나로 모델의 부품을 만들어요.


## 3.5 글자 임베딩: 번호를 벡터로

번호 그 자체는 의미가 없어요('가'가 102번, '각'이 103번이라고 둘이 비슷한 건 아니죠). 그래서 글자마다 **숫자 여러 개로 된 벡터**를 하나씩 주고, 학습하면서 이 벡터를 고쳐 나갑니다.
`nn.Embedding`은 번호를 넣으면 그 번호의 벡터를 꺼내 주는 **찾아보기 표**예요. 지금은 학습 전이라 무작위 값입니다.

**핵심 코드**: `nn.Embedding(len(vocab), n_embd)` — 글자 수 × 벡터 길이 크기의 표


In [ ]:
n_embd = 16
token_emb = nn.Embedding(len(vocab), n_embd)
print("찾아보기 표 모양:", token_emb.weight.shape)

i = vocab.stoi["요"]
numbers = []
for v in token_emb.weight[i][:5].tolist():
    numbers.append(round(v, 2))
print("'요'의 벡터 앞 5칸:", numbers)


이번에는 3.1의 조각 `x`(8글자)를 통째로 표에 넣어 봅니다. 번호 하나하나가 벡터 하나로 바뀌어요.


In [ ]:
tok = token_emb(x)
print("x 모양:", x.shape)
print("임베딩 모양:", tok.shape)


## 3.6 위치 임베딩: 몇 번째 글자인가

같은 글자라도 어디에 있느냐에 따라 역할이 달라요. 그래서 **자리마다** 벡터를 하나씩 두고 글자 벡터에 더합니다.

**핵심 코드**: `tok + p` — (8, 16) + (8, 16)


In [ ]:
pos_emb = nn.Embedding(block_size, n_embd)

pos = torch.arange(block_size)
print("자리 번호:", pos)

p = pos_emb(pos)
print("자리 벡터 모양:", p.shape)

x_emb = tok + p
print("더한 결과 모양:", x_emb.shape)


## 3.7 부품으로 묶기: Embeddings 클래스

3.5와 3.6에서 만든 표 두 개와 더하기를 **부품 하나**로 묶습니다.
먼저 2장에서 배운 **함수**로 묶어 보고, 함수로는 아쉬운 점을 **클래스**로 풀어요.

**핵심 코드**: `class Embeddings(nn.Module):`


In [ ]:
def embed(x):
    tok = token_emb(x)
    pos = torch.arange(len(x))
    p = pos_emb(pos)
    return tok + p

print(torch.equal(embed(x), x_emb))


함수 안의 `token_emb`와 `pos_emb`는 함수 **바깥**에서 만든 표예요. 표와 계산을 한 상자에 담는 문법이 클래스입니다. PyTorch 없이 작은 클래스부터 만들어 봅니다.


In [ ]:
class FishBread:
    def __init__(self, filling):
        self.filling = filling

    def describe(self):
        return self.filling + " 붕어빵"

bread1 = FishBread("팥")
bread2 = FishBread("슈크림")
print(bread1.describe())
print(bread2.describe())
print(bread1.filling)


붕어빵 틀과 같은 모양으로, 모델의 크기를 정하는 설정 다섯 개를 담는 상자 `GPTConfig`를 만듭니다. 값마다 기본값을 주어서, 바꾸고 싶은 값만 골라 넣을 수 있어요.


In [ ]:
class GPTConfig:
    def __init__(self, vocab_size=2453, block_size=384, n_embd=256,
                 n_head=4, n_layer=6):
        self.vocab_size = vocab_size    # 사전의 글자 수 (2장)
        self.block_size = block_size    # 한 번에 보는 글자 수 (3.1)
        self.n_embd = n_embd            # 글자 벡터의 길이 (3.5)
        self.n_head = n_head            # 어텐션 헤드 수 (5장에서 씀)
        self.n_layer = n_layer          # 블록 층 수 (6장에서 씀)

cfg = GPTConfig(vocab_size=len(vocab), block_size=8, n_embd=16)
print(cfg.vocab_size, cfg.block_size, cfg.n_embd, cfg.n_head, cfg.n_layer)


같은 모양으로 `Embeddings` 부품을 만들고, 방금 만든 설정 상자 `cfg`를 넘겨요. 아래쪽 줄들은 **부품을 부르면 forward가 실행된다**는 것, 부품 안에 무엇이 붙어 있는지, 그리고 부품마다 **자기 표**를 따로 가진다는 것을 확인합니다.


In [ ]:
class Embeddings(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)

    def forward(self, idx):
        T = idx.shape[-1]
        pos = torch.arange(T, device=idx.device)
        tok = self.token_emb(idx)
        p = self.pos_emb(pos)
        return tok + p

emb = Embeddings(cfg)
out = emb(x)
print("임베딩 결과 모양:", out.shape)
print(torch.equal(emb(x), emb.forward(x)))
print(emb)

emb2 = Embeddings(cfg)                  # 부품 하나 더 만들기
print(torch.equal(emb(x), emb2(x)))


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

학습 없이 **세어 보기**만으로 다음 글자를 맞혀 봅시다. 어떤 글자 바로 뒤에 무슨 글자가 몇 번 나왔는지 세는 거예요.
"토" 하나만 보고는 "토끼"인지 "토리"인지 알 수 없습니다. **앞 글자 하나만으로는 부족하다**는 것, 이것이 4장 어텐션의 출발점입니다.


In [ ]:
from collections import Counter

def count_next(ch):
    counts = Counter()
    for i in range(len(text) - 1):
        if text[i] == ch:
            nxt = text[i + 1]
            counts[nxt] = counts[nxt] + 1
    return counts

for ch in ["친", "요", "토"]:
    counts = count_next(ch)
    total = sum(counts.values())
    print(f"{ch} 다음 (모두 {total}번)")
    for b, c in counts.most_common(5):
        percent = round(c / total * 100)
        print(f"  [{b}] {c}번, {percent}%")


## 실제 LLM에서는

- 실제 LLM의 임베딩 벡터 길이는 수천에 이릅니다.
- 위치 정보도 자리마다 벡터를 더하는 대신, 어텐션 계산 안에서 벡터를 회전시키는 방식(회전 위치 인코딩 등)을 많이 씁니다. 그래야 학습 때보다 긴 글에도 잘 대응해요.
- 놀이터의 "세어 보기"는 오래전에 쓰이던 n-그램 언어 모델 방식입니다.
